In [4]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import PydanticOutputParser
from langchain_tavily import TavilySearch
from langchain_core.runnables import RunnableLambda
import os
from datetime import datetime, timedelta
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from rich import print as rprint


load_dotenv(dotenv_path='.env')

# 1. 初始化组件
web_search = TavilySearch(max_results=4, api_key=os.getenv("TAVILY_API_KEY"))
model = init_chat_model(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-v4-pro",
    model_provider="deepseek",
    # temperature=0,
    base_url = "https://api.deepseek.com"
)
checkpointer = InMemorySaver()

agent = create_agent(
    model = model,
    checkpointer = checkpointer,
)
config = {
    "configurable": {
        "thread_id": "Enfield",
    }
}

# 2. 定义搜索函数（使用 invoke 方法）
# @tool()
# def search_movie(query):
#     """搜索电影信息（正确调用方式）"""
#     # 使用 invoke() 方法，传入字典参数
#     return web_search.invoke({"query": query})

response1 = agent.invoke(
    {"messages": [HumanMessage(content = "你好，我是Enfield")]},
    config= config)
response2 = agent.invoke({
    "messages": [HumanMessage(content = "请问我叫什么？")]},
    config = config)

print(response2["messages"][-1].content)


你刚才告诉我，你叫 **Enfield** 呀～ 😊 我记着呢。


## 第二段
### 数据库级长期记忆

In [ ]:
from langgraph.checkpoint.mysql import BaseMySQLSaver
from langgraph.graph import StateGraph, MessagesState

